# Vector Databases: Chroma DB & Comparative Analysis
This notebook introduces **Chroma DB**, an open-source native vector database for AI applications. It compares Chroma's integrated document/metadata storage architecture against FAISS, and demonstrates end-to-end PDF ingestion, persistence, LCEL RAG execution, and metadata filtering.

---

## FAISS vs Chroma

| Feature | **FAISS** | **Chroma** |
|---------|-----------|------------|
| Flat Index | ✅ | ❌ |
| IVF | ✅ | ❌ |
| HNSW | ✅ | ✅ |
| Product Quantization (PQ) | ✅ | ❌ |
| Manual Index Selection | ✅ | ❌ |
| Metadata Filtering | Limited (via wrappers/applications) | ✅ Native |
| Persistence | Manual save/load | ✅ Built-in |
| LangChain Integration | ✅ | ✅ |

---

## Final Understanding

### FAISS = Vector Index / Similarity Search Library

With FAISS, the developer is responsible for managing:

- Index type
- Embedding dimension
- Similarity metric
- Document storage
- ID mapping
- Persistence

---

### Chroma = Vector Database

Chroma automatically manages:

- Embeddings (vectors)
- Documents
- Metadata
- IDs
- Collections
- Indexing
- Persistence

---

## FAISS

FAISS is a **low-level, high-performance library** developed by Meta for dense-vector similarity search and clustering.

It gives developers direct control over:

- Flat indexes
- IVF indexes
- HNSW indexes
- Product Quantization (PQ)
- Other ANN index structures
- CPU and GPU acceleration

However, **FAISS is not a complete vector database.**

Features such as:

- Document storage
- Metadata management
- Filtering
- CRUD operations
- Collections
- Persistence orchestration
- Client/server APIs

must generally be implemented separately or through another library.

---

## Chroma

Chroma is a **vector database** designed for Retrieval-Augmented Generation (RAG) and AI applications.

A Chroma collection stores:

- Embeddings
- Documents
- Metadata
- IDs

It also provides:

- Collections
- Metadata filtering
- CRUD operations
- Persistence
- Client/server deployment
- Hosted cloud deployment

Current versions of Chroma primarily use **HNSW** for vector search while also supporting additional retrieval indexes (such as SPANN and sparse/full-text retrieval) within its broader retrieval architecture.

---

## When Should You Use Each?

### Choose FAISS when:

- Maximum search performance is required.
- You need full control over ANN algorithms.
- You want to choose the index type yourself.
- GPU acceleration is important.
- You need compression techniques such as PQ.

---

### Choose Chroma when:

- Building a complete RAG application.
- Documents and metadata should be stored together.
- Metadata filtering is required.
- CRUD operations are needed.
- You want persistence without managing multiple files.
- You prefer a database-style interface.

---

## Does FAISS Store Documents?

A common misconception is that FAISS stores documents and metadata.

For example, in LangChain code such as:

```python
vector_store = FAISS(
    embedding_function=embeddings,
    index=faiss_index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={}
)
```

it appears that FAISS stores everything.

**However, this is not what actually happens.**

The **FAISS library only stores numerical embedding vectors.**

The **LangChain FAISS wrapper** stores:

- Documents
- Metadata
- ID mappings

using additional Python objects.

---

## Native FAISS vs LangChain FAISS Wrapper

```text
LangChain FAISS VectorStore
│
├── FAISS Index
│     └── Numerical embedding vectors
│
├── InMemoryDocstore
│     └── LangChain Document objects
│           ├── page_content
│           └── metadata
│
└── index_to_docstore_id
      └── Maps FAISS vector positions to Document IDs
```

---

## Chroma Storage Model

In Chroma, documents, metadata, IDs, and embeddings are stored together inside the same collection.

Example:

```python
collection.add(
    ids=["chunk-1"],
    documents=["Llama 2 is a family of language models."],
    metadatas=[
        {
            "source": "llama2.pdf",
            "page": 5
        }
    ],
    embeddings=[[0.1, 0.2, 0.3]]
)
```

Unlike FAISS, no external document store or ID mapping is required.

---

## Feature Comparison

| Feature | **FAISS** | **Chroma** |
|---------|-----------|------------|
| Store Vectors | ✅ | ✅ |
| Store Documents | ❌ (Native) | ✅ |
| Store Metadata | ❌ (Native) | ✅ |
| Collections | ❌ | ✅ |
| CRUD Operations | Limited | ✅ |
| Metadata Filtering | ❌ (Native) | ✅ |
| Persistence | Basic index save/load | ✅ |
| Client APIs | ❌ | ✅ |
| Server Mode | ❌ | ✅ |

---

## LangChain + FAISS vs Chroma

| Component | **LangChain + FAISS** | **Chroma** |
|-----------|-----------------------|------------|
| Embeddings | Native FAISS index | Chroma vector index |
| Documents | LangChain `Docstore` | Chroma collection |
| Metadata | `Document.metadata` | Collection record |
| ID Mapping | `index_to_docstore_id` | Managed internally |
| Saving | FAISS index + Pickle files | Database persistence |
| Metadata Filtering | Wrapper/application logic | Native filtering |
| Collections | Not native to FAISS | Native |
| CRUD Operations | Wrapper/index-dependent | Native |
| Server / Cloud | Separate infrastructure required | Built-in database architecture |

---

## Simple Classroom Explanation

Think of FAISS as a **high-performance search engine for vectors**.

It focuses on:

- Storing vectors
- Finding nearest neighbors
- Providing fast ANN algorithms

Everything else—documents, metadata, IDs, filtering, and persistence—must be handled by you or by a wrapper such as LangChain.

Think of Chroma as a **database built specifically for AI applications**.

It stores everything (vectors, documents, metadata, and IDs) together and provides database features like filtering, CRUD operations, persistence, and collections out of the box.

> **In short:** FAISS is a **vector indexing/search library**, while Chroma is a **vector database** that uses vector indexes internally.

---

#### Official Chroma Website
- https://www.trychroma.com/

## 1. Setup & Library Imports
Import required components from `langchain_chroma`, `langchain_community`, `langchain_google_genai`, and LCEL core modules.

In [8]:
# Load required libraries for Chroma DB, document loading, text splitting, and LCEL
from dotenv import load_dotenv
import os

from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings,
    ChatGoogleGenerativeAI
)

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

## 2. Environment Setup
Load API keys from the `.env` file to enable Google Generative AI embeddings and LLM inference.

In [9]:
# Load .env file and set GOOGLE_API_KEY in environment
# Load the .env file and set the GOOGLE_API_KEY explicitly in the environment
load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

## 3. Initialize Embedding Model
Instantiate `GoogleGenerativeAIEmbeddings` using the `gemini-embedding-001` model.

In [10]:
# Initialize Google Generative AI embedding model
# Instantiate the embedding model
embedding_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

## 4. Document Ingestion & Chunking
Load the PDF file (`llama2-research-paper.pdf`) using `PyPDFLoader` and split pages into semantic chunks using `RecursiveCharacterTextSplitter`.

In [11]:
# Load pages from PDF file
file_path = r"D:\Coding\Full-Stack-GenAI-AgenticAI-Bootcamp\05_RAG\03_Vector_Databases\data\llama2-research-paper.pdf"
loader = PyPDFLoader(file_path)
pages = loader.load()
print("Total pages:", len(pages))

Total pages: 77


In [12]:
# Configure text splitter and generate document chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)
chunks = text_splitter.split_documents(pages)
print("Total chunks:", len(chunks))

Total chunks: 175


## 5. Initialize Chroma Vector Store
Unlike FAISS, **Chroma DB** natively stores text documents, metadata, and vectors in unified collections. We set `persist_directory` for built-in persistence and specify `hnsw:space: cosine` for similarity search.

In [13]:
# Create a persistent Chroma collection named 'llama2_collection' using Cosine distance
vector_store = Chroma(
    collection_name="llama2_collection",
    embedding_function=embedding_model,
    persist_directory="./chroma_db_llama2",
    collection_metadata={
        "hnsw:space": "cosine"
    }
)

## 6. Batch Document Ingestion with Rate Limiting
Insert document chunks into the Chroma store in small batches, handling 429 rate limit exceptions automatically.

In [15]:
# Add documents in batches with retry logic for rate limits
import time

batch_size = 20
total_added = 0

for i in range(0, len(chunks), batch_size):
    print(f"Adding batch {i} to {i+batch_size} of {len(chunks)}...")
    batch = chunks[i:i+batch_size]
    
    success = False
    while not success:
        try:
            added_ids = vector_store.add_documents(documents=batch)
            total_added += len(added_ids)
            success = True
            if i + batch_size < len(chunks):
                time.sleep(15)
        except Exception as e:
            error_msg = str(e)
            if "429" in error_msg or "RESOURCE_EXHAUSTED" in error_msg:
                print("Rate limit hit! Pausing for 60 seconds before retrying...")
                time.sleep(60)
            else:
                raise e

print("Documents added:", total_added)
print(
    "Total documents stored:",
    vector_store._collection.count()
)

Adding batch 0 to 20 of 175...
Adding batch 20 to 40 of 175...
Adding batch 40 to 60 of 175...
Adding batch 60 to 80 of 175...
Rate limit hit! Pausing for 60 seconds before retrying...
Adding batch 80 to 100 of 175...
Adding batch 100 to 120 of 175...
Adding batch 120 to 140 of 175...
Adding batch 140 to 160 of 175...
Adding batch 160 to 180 of 175...
Documents added: 175
Total documents stored: 175


## 7. Basic Retrieval & Query Testing
Convert the Chroma vector store into a retriever and test document similarity search.

In [16]:
# Create a retriever to fetch top 5 (k=5) most relevant chunks
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5
    }
)

In [17]:
# Test retriever with a sample query and print retrieved content along with metadata
query = "What is the architecture of Llama 2?"

retrieved_documents = retriever.invoke(query)

for i, document in enumerate(
    retrieved_documents,
    start=1
):
    print(f"\n--- Retrieved document {i} ---")
    print(document.page_content[:500])
    print("Metadata:", document.metadata)


--- Retrieved document 1 ---
capabilities and limitations of our models; results can be found in Section 4.1.
2.2 Training Details
We adopt most of the pretraining setting and model architecture fromLlama 1. We use the standard
transformer architecture (Vaswani et al., 2017), apply pre-normalization using RMSNorm (Zhang and
Sennrich, 2019), use the SwiGLU activation function (Shazeer, 2020), and rotary positional embeddings
(RoPE, Su et al. 2022). The primary architectural differences fromLlama 1 include increased context l
Metadata: {'trapped': '/False', 'page': 4, 'title': '', 'total_pages': 77, 'producer': 'pdfTeX-1.40.25', 'page_label': '5', 'keywords': '', 'source': 'D:\\Coding\\Full-Stack-GenAI-AgenticAI-Bootcamp\\05_RAG\\03_Vector_Databases\\data\\llama2-research-paper.pdf', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'moddate': '2023-07-20T00:30:36+00:00', 'subject': '', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.

## 8. Build End-to-End RAG Pipeline with LCEL
Construct a Retrieval-Augmented Generation chain combining the Chroma retriever, prompt template, Gemini LLM (`gemini-3.1-flash-lite`), and output parser.

In [18]:
# Define ChatPromptTemplate for grounded Q&A
prompt = ChatPromptTemplate.from_template(
    """
    You are a question-answering assistant.

    Answer the question only from the provided context.

    If the context does not contain the answer, say:
    "I do not have enough information in the provided document."

    Context:
    {context}

    Question:
    {question}

    Answer:
    """
)

In [19]:
# Define document formatting helper that includes source and page number metadata
def format_docs(docs):
    return "\n\n".join(
        f"""
        Source: {doc.metadata.get("source")}
        Page: {doc.metadata.get("page")}

        {doc.page_content}
        """
        for doc in docs
    )

In [20]:
# Initialize the LLM (gemini-3.1-flash-lite)
model = ChatGoogleGenerativeAI(
    model = 'gemini-3.1-flash-lite',
    temperature=0
)

In [21]:
# Construct the LCEL RAG chain
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | model
    | StrOutputParser()
)

In [22]:
# Invoke the RAG pipeline to generate an answer grounded in the PDF context
answer = rag_chain.invoke(
    "What is the architecture of Llama 2?"
)

print("\nFinal answer:\n")
print(answer)


Final answer:

Llama 2 is an auto-regressive language model that uses an optimized transformer architecture. It applies pre-normalization using RMSNorm, uses the SwiGLU activation function, and rotary positional embeddings (RoPE). Key architectural features include increased context length and the use of grouped-query attention (GQA) for larger models.


## 9. Loading Persisted Chroma DB from Disk
Since Chroma DB automatically saves data to `persist_directory`, we can reconnect to an existing collection across sessions without re-embedding.

In [23]:
# Load existing Chroma collection from local disk directory
from langchain_chroma import Chroma

loaded_vector_store = Chroma(
    collection_name="llama2_collection",
    embedding_function=embedding_model,
    persist_directory="./chroma_db_llama2"
)

In [24]:
# Create a retriever from the reloaded Chroma vector store
loaded_retriever = loaded_vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

In [25]:
# Query the reloaded store to verify data persistence
docs = loaded_retriever.invoke(
    "What is Llama 2?"
)

for doc in docs:
    print(doc.page_content[:500])

Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev
Figure 3: Safety human evaluation results forLlama 2-Chat compared to other open-source and closed-
source models. Human raters judged model generations for safety violations across ~2,000 adversarial
prompts consisting of both single and multi-turn prompts. More details can be found in Section 4.4. It is
important to caveat these safety results with the inherent bias of LLM evaluations due to limitations of the
prompt set, subjectivity of the review guidelines, and subjectivity of individual r

## 10. Native Metadata Filtering
Chroma DB supports native metadata filtering. We can narrow search results down to specific pages, authors, or sources during retrieval.

In [26]:
# Explicitly persist vector store state to disk
vector_store.persist()

AttributeError: 'Chroma' object has no attribute 'persist'

In [27]:
# Create a retriever configured with a metadata filter (e.g., page == 10)
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5,
        "filter": {
            "page": 10
        }
    }
)

In [ ]:
# Perform similarity search with metadata filter for page 10
results = vector_store.similarity_search(
    query="What is reinforcement learning?",
    k=5,
    filter={
        "page": 10
    }
)

In [31]:
# Display filtered similarity search results
print(results)

[Document(id='15aa00ab-9d38-4e23-bb79-ebaa40523ef9', metadata={'page_label': '11', 'subject': '', 'author': '', 'producer': 'pdfTeX-1.40.25', 'creationdate': '2023-07-20T00:30:36+00:00', 'source': 'D:\\Coding\\Full-Stack-GenAI-AgenticAI-Bootcamp\\05_RAG\\03_Vector_Databases\\data\\llama2-research-paper.pdf', 'title': '', 'total_pages': 77, 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'creator': 'LaTeX with hyperref', 'page': 10, 'moddate': '2023-07-20T00:30:36+00:00', 'keywords': '', 'trapped': '/False'}, page_content='score than its counterpart. We used a binary ranking loss consistent with Ouyang et al. (2022):\nLranking = −log(σ(rθ(x, yc) − rθ(x, yr))) (1)\nwhere rθ(x, y) is the scalar score output for promptx and completiony with model weightsθ. yc is the\npreferred response that annotators choose andyr is the rejected counterpart.\nBuilt on top of this binary ranking loss, we further modify it separately for better he